# Task 2 · Contracts, nested data and a star schema

**Daniel Ehab** · 58-3825 · MET-02 · CSEN1095 Data Engineering, W26

The student clubs sell tickets to their events through an app, and the app sends every booking as one JSON document per line. This notebook keeps only the bookings that meet the contract (Part A), flattens them into tables (Part B), and models them as a star schema in DuckDB, saved as Parquet (Part C).

The input file is set once, in the first code cell, and every later cell reads it through `DATA`. Nothing here edits the file, and every count is computed from it, so the notebook runs unchanged on another file with the same shape.

In [1]:
DATA = "bookings.jsonl"

## Setup

The course's Python 3.12, with the packages pinned in `requirements.txt`: the same versions as Labs 4 and 5. The versions are printed so that anyone who re-runs the notebook can compare.

In [2]:
import datetime as dt
import json
import math
import re
import sys
from collections import Counter
from typing import Literal

import duckdb
import ipykernel
import pandas as pd
import pyarrow
import pydantic
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator

pd.set_option("display.max_colwidth", 120)  # reasons are long; show them in full

print("Python    ", sys.version.split()[0])
for module in [ipykernel, pd, pydantic, duckdb, pyarrow]:
    print(f"{module.__name__:<10}", module.__version__)

Python     3.12.14
ipykernel  7.3.0
pandas     2.3.3
pydantic   2.13.5
duckdb     1.5.6
pyarrow    26.0.0


## Part A · The contract

### A.0 Look first

The brief shows one clean document. Before writing a single rule, we read every line of `DATA` and answer the first-look questions for nested data: what one line is, which keys it has, and which values each field takes.

The file is read once, as text, into `lines`. Nothing assigns to `lines` after this cell; every later step starts from it. Each line is parsed on its own, so a line that isn't valid JSON is counted instead of stopping the notebook, and every document keeps its line number.

In [3]:
with open(DATA, encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]

docs = {}      # line number -> the parsed JSON value
not_json = {}  # line number -> the parser's error message
for n, line in enumerate(lines, start=1):
    try:
        docs[n] = json.loads(line)
    except json.JSONDecodeError as e:
        not_json[n] = str(e)

print("lines read:       ", len(lines))
print("parsed as JSON:   ", len(docs))
print("not valid JSON:   ", len(not_json), not_json)
print("not a JSON object:", sum(not isinstance(d, dict) for d in docs.values()))

lines read:        283
parsed as JSON:    283
not valid JSON:    0 {}
not a JSON object: 0


**One document**, the first one that parsed, printed with indentation so its nesting shows:

In [4]:
first = next(iter(docs.values()))
print(json.dumps(first, indent=2))

{
  "booking_id": "B0001",
  "booked_at": "2026-09-28T15:46:00Z",
  "student": {
    "id": "60-3778",
    "name": "Nour Mansour",
    "faculty": "IET"
  },
  "event": {
    "id": "EV-01",
    "title": "Robotics Demo Day",
    "club": "Robotics",
    "date": "2026-10-15",
    "venue": "Hall A"
  },
  "tickets": [
    {
      "type": "standard",
      "qty": 1,
      "price": 40.0
    },
    {
      "type": "vip",
      "qty": 3,
      "price": 100.0
    }
  ],
  "total": 340.0,
  "payment": "card",
  "status": "cancelled"
}


**Four small helpers** for looking inside the documents without trusting their shape:

- `at(doc, "student", "faculty")` walks a path and returns `"<no key>"` when a key is missing, so a missing key and a `null` value stay different;
- `tickets_of(doc)` returns the ticket list, or an empty list when `tickets` isn't a list;
- `shape(value)` describes a value: an object's sorted keys, or the type of anything else;
- `show(title, counter)` prints a `Counter`, most common first.

Values are counted by their `repr`, so text shows with its quotes and a hidden space can't hide (Task 1's `'Music '`).

In [5]:
def at(doc, *path):
    """The value at a path inside a document, or "<no key>" if a step is missing."""
    for key in path:
        if not isinstance(doc, dict) or key not in doc:
            return "<no key>"
        doc = doc[key]
    return doc


def tickets_of(doc):
    """The document's ticket list, or an empty list if it doesn't have one."""
    tickets = at(doc, "tickets")
    return tickets if isinstance(tickets, list) else []


def shape(value):
    """An object's sorted keys, or the type name of anything else ('NoneType' for null)."""
    return tuple(sorted(value)) if isinstance(value, dict) else type(value).__name__


def show(title, counter):
    print(title)
    for value, n in counter.most_common():
        print(f"  {n:>4}  {value}")


objects = [d for d in docs.values() if isinstance(d, dict)]
tickets = [t for d in objects for t in tickets_of(d)]
print(len(objects), "documents,", len(tickets), "ticket entries")

283 documents, 344 ticket entries


**Which keys does each level have?** Counted on the raw dicts, not on a DataFrame: once flattened, a missing key and a `null` look the same.

In [6]:
show("top level", Counter(shape(d) for d in objects))
show("student", Counter(shape(at(d, "student")) for d in objects))
show("event", Counter(shape(at(d, "event")) for d in objects))
show("tickets", Counter(shape(at(d, "tickets")) for d in objects))
show("each ticket", Counter(shape(t) for t in tickets))

top level
   193  ('booked_at', 'booking_id', 'event', 'payment', 'promo_code', 'status', 'student', 'tickets', 'total')
    90  ('booked_at', 'booking_id', 'event', 'payment', 'status', 'student', 'tickets', 'total')
student
   283  ('faculty', 'id', 'name')
event
   281  ('club', 'date', 'id', 'title', 'venue')
     2  NoneType
tickets
   283  list
each ticket
   344  ('price', 'qty', 'type')


**The fields with a fixed list of allowed values**, and how many tickets each booking holds:

In [7]:
show("student.faculty", Counter(repr(at(d, "student", "faculty")) for d in objects))
show("event.club", Counter(repr(at(d, "event", "club")) for d in objects))
show("ticket type", Counter(repr(at(t, "type")) for t in tickets))
show("payment", Counter(repr(at(d, "payment")) for d in objects))
show("status", Counter(repr(at(d, "status")) for d in objects))
show("tickets per booking", Counter(len(tickets_of(d)) for d in objects))

student.faculty
    75  'EMS'
    62  'MET'
    60  'IET'
    53  'Pharmacy'
    33  'Management'
event.club
    76  'Debate'
    74  'Music'
    56  'Robotics'
    40  'Chess'
    35  'Football'
     2  '<no key>'
ticket type
   187  'standard'
    78  'vip'
    76  'student'
     2  'early-bird'
     1  'premium'
payment
   145  'card'
    70  'cash'
    68  'wallet'
status
   257  'confirmed'
    26  'cancelled'
tickets per booking
   218  1
    63  2
     2  0


**The numbers**: their types, their range, and every value of `qty`.

In [8]:
def numbers(values):
    return [v for v in values if isinstance(v, (int, float)) and not isinstance(v, bool)]


for name, values in [("qty", [at(t, "qty") for t in tickets]),
                     ("price", [at(t, "price") for t in tickets]),
                     ("total", [at(d, "total") for d in objects])]:
    types = dict(Counter(type(v).__name__ for v in values))
    nums = numbers(values)
    print(f"{name:<6} types {types}  min {min(nums, default=None)}  max {max(nums, default=None)}")

show("qty values", Counter(repr(at(t, "qty")) for t in tickets))

qty    types {'int': 344}  min -1  max 4
price  types {'float': 344}  min 10.0  max 200.0
total  types {'float': 283}  min -30.0  max 850.0
qty values
   140  1
    99  2
    53  3
    49  4
     2  -1
     1  0


**Ids and dates**: each value with every digit replaced by `9`, so values of the same format collapse into one line.

In [9]:
def digits_as_9(value):
    return re.sub(r"\d", "9", value) if isinstance(value, str) else repr(value)


show("booking_id", Counter(digits_as_9(at(d, "booking_id")) for d in objects))
show("student.id", Counter(digits_as_9(at(d, "student", "id")) for d in objects))
show("booked_at", Counter(digits_as_9(at(d, "booked_at")) for d in objects))
show("event.date", Counter(digits_as_9(at(d, "event", "date")) for d in objects))

booking_id
   283  B9999
student.id
   279  99-9999
     2  99-99999
     1  999999
     1  99_9999
booked_at
   283  9999-99-99T99:99:99Z
event.date
   281  9999-99-99
     2  <no key>


**Does `booking_id` repeat?** The brief warns that the app sometimes sends a booking twice.

In [10]:
ids = Counter(repr(at(d, "booking_id")) for d in objects)
repeated = {i: n for i, n in ids.items() if n > 1}
print("documents:", len(objects), "| distinct booking_id:", len(ids), "| ids that repeat:", len(repeated))
print(repeated)

documents: 283 | distinct booking_id: 280 | ids that repeat: 3
{"'B0035'": 2, "'B0066'": 2, "'B0107'": 2}


**What we see** in `bookings.jsonl`, and where each finding is handled. The counts are in the outputs above; this cell names none, because a file with different problems would change them.

| The output shows | The brief's rule | Handled in |
|---|---|---|
| Every line parses as a JSON object | Validate every line | A.2 still quarantines a line that doesn't |
| Two document shapes: some carry `promo_code`, which the example lacks | Part B.3: find the drifting field | A.1 declares it, B.3 finds when it appears |
| Some documents have `"event": null` | `event` present, with all five fields | A.1, booking model |
| Some bookings have an empty ticket list | At least one ticket | A.1, booking model |
| Ticket types `'early-bird'` and `'premium'` | `type` one of `standard`, `vip`, `student` | A.1, ticket model |
| `qty` of `0` and `-1` | `qty` at least 1 | A.1, ticket model |
| `student.id` in three other formats (`99-99999`, `999999`, `99_9999`) | Two digits, a dash, four digits | A.1, student model |
| A negative `total` | `total` equals the sum of `qty × price` | A.1's model validator decides |
| Faculty, club, payment and status hold only allowed values, spelled exactly | One of the allowed values | A.1 writes the rules anyway, for the second file |
| `booked_at` has one format, ISO 8601 in UTC with a `Z`; `event.date` is always `YYYY-MM-DD` | `booked_at` before the event's date | A.1 types them as a datetime and a date |
| Some `booking_id`s appear twice | Keep the first copy | A.3 |
| Most bookings hold one ticket entry, some hold two | | B.1: `booking_tickets` has more rows than `bookings` |

Two rules can't be seen by counting values: whether each total adds up, and whether anyone booked after the event. Both compare fields inside one booking, so they wait for A.1's model validator.

### A.1 The contract as Pydantic models

The brief's table is the contract. Each rule becomes a type or a `Field` constraint, so it is written once, in a model, and Pydantic checks it on every document.

The rules come from the brief, not from what A.0 found in this file, because the graders run the notebook again on a file with different problems. What A.0 found becomes probes instead: broken values we feed to the models to watch each rule refuse them.

Pydantic runs in its default **lax mode** (Lab 4): it converts a value when nothing is lost (`"2"` becomes `2` for an `int`) and refuses when something would be (`2.5`, `"two"`).

First the three small models. The booking model that uses them, with the two rules that compare fields, comes after them.

| The brief's rule | Where it lives |
|---|---|
| `student.id`: two digits, a dash, four digits | `Student.id`, `pattern=r"^\d{2}-\d{4}$"` |
| `student.faculty`: one of `MET`, `IET`, `EMS`, `Pharmacy`, `Management` | `Student.faculty`, the `Faculty` literal |
| `event`: all five fields; `club` one of `Robotics`, `Debate`, `Music`, `Football`, `Chess` | `Event`: five required fields, `club` the `Club` literal |
| Each ticket: `type` one of `standard`, `vip`, `student`; `qty` at least 1; `price` not negative | `Ticket`: the `TicketType` literal, `ge=1`, `ge=0` |
| `event` present, at least one ticket, `total`, `booked_at`, `payment`, `status` | The booking model, next |

In [11]:
Faculty = Literal["MET", "IET", "EMS", "Pharmacy", "Management"]
Club = Literal["Robotics", "Debate", "Music", "Football", "Chess"]
TicketType = Literal["standard", "vip", "student"]


class Ticket(BaseModel):
    type: TicketType
    qty: int = Field(ge=1)      # at least 1
    price: float = Field(ge=0)  # not negative, so a free ticket is allowed


class Student(BaseModel):
    id: str = Field(pattern=r"^\d{2}-\d{4}$")  # two digits, a dash, four digits, and nothing else
    name: str
    faculty: Faculty


class Event(BaseModel):
    id: str
    title: str
    club: Club
    date: dt.date  # "2026-10-15" becomes a real date, so the booking model can compare it
    venue: str

**Probes**, like Lab 4's check cells. The good examples are the brief's own example document; each probe then breaks one rule, and Pydantic must refuse it and name that field. The broken values include the ones A.0 found in this file (`'early-bird'`, `'premium'`, a `qty` of `0` and `-1`, the three wrong id formats) and others it didn't.

`refused(model, data)` returns the fields Pydantic names when it refuses the data, or `None` when it accepts it.

In [12]:
def refused(model, data):
    """The fields Pydantic names when it refuses data, or None when it accepts it."""
    try:
        model.model_validate(data)
        return None
    except ValidationError as e:
        return sorted({".".join(map(str, err["loc"])) for err in e.errors()})


good_ticket = {"type": "standard", "qty": 1, "price": 40.0}
good_student = {"id": "61-3826", "name": "Laila Adel", "faculty": "IET"}
good_event = {"id": "EV-01", "title": "Robotics Demo Day", "club": "Robotics", "date": "2026-10-15", "venue": "Hall A"}

PROBES = [  # (what the probe does, model, data, the field Pydantic must name, or None if it must pass)
    ("the brief's ticket", Ticket, good_ticket, None),
    ("type 'early-bird'", Ticket, {**good_ticket, "type": "early-bird"}, "type"),
    ("type 'premium'", Ticket, {**good_ticket, "type": "premium"}, "type"),
    ("type 'VIP', in capitals", Ticket, {**good_ticket, "type": "VIP"}, "type"),
    ("qty 0", Ticket, {**good_ticket, "qty": 0}, "qty"),
    ("qty -1", Ticket, {**good_ticket, "qty": -1}, "qty"),
    ("qty 2.5", Ticket, {**good_ticket, "qty": 2.5}, "qty"),
    ("qty '2', as text", Ticket, {**good_ticket, "qty": "2"}, None),
    ("price 0, a free ticket", Ticket, {**good_ticket, "price": 0}, None),
    ("price -5", Ticket, {**good_ticket, "price": -5.0}, "price"),
    ("ticket without price", Ticket, {"type": "vip", "qty": 1}, "price"),
    ("the brief's student", Student, good_student, None),
    ("id '61-38261', a fifth digit", Student, {**good_student, "id": "61-38261"}, "id"),
    ("id '613826', no dash", Student, {**good_student, "id": "613826"}, "id"),
    ("id '61_3826', an underscore", Student, {**good_student, "id": "61_3826"}, "id"),
    ("id 'ID61-3826', text in front", Student, {**good_student, "id": "ID61-3826"}, "id"),
    ("faculty 'iet', lower case", Student, {**good_student, "faculty": "iet"}, "faculty"),
    ("the brief's event", Event, good_event, None),
    ("event without venue", Event, {k: v for k, v in good_event.items() if k != "venue"}, "venue"),
    ("venue null", Event, {**good_event, "venue": None}, "venue"),
    ("club 'Chess Club'", Event, {**good_event, "club": "Chess Club"}, "club"),
    ("date '15/10/2026'", Event, {**good_event, "date": "15/10/2026"}, "date"),
]

probe_results = pd.DataFrame([
    {"probe": label, "model": model.__name__, "must name": must or "(passes)",
     "Pydantic names": ", ".join(refused(model, data) or ["(passes)"])}
    for label, model, data, must in PROBES
])
probe_results["ok"] = probe_results["must name"] == probe_results["Pydantic names"]
probe_results

,probe,model,must name,Pydantic names,ok
0,the brief's ticket,Ticket,(passes),(passes),True
1,type 'early-bird',Ticket,type,type,True
2,type 'premium',Ticket,type,type,True
3,"type 'VIP', in capitals",Ticket,type,type,True
4,qty 0,Ticket,qty,qty,True
5,qty -1,Ticket,qty,qty,True
6,qty 2.5,Ticket,qty,qty,True
7,"qty '2', as text",Ticket,(passes),(passes),True
8,"price 0, a free ticket",Ticket,(passes),(passes),True
9,price -5,Ticket,price,price,True


In [13]:
# ✓ check
assert probe_results["ok"].all(), probe_results[~probe_results["ok"]]
print("✓", len(PROBES), "probes: each good example passes, and each broken rule is refused for its own field")

✓ 22 probes: each good example passes, and each broken rule is refused for its own field


**Why `^` and `$` in the id pattern?** The same pattern without them, on two ids that break the rule:

In [14]:
class StudentWithoutAnchors(BaseModel):
    id: str = Field(pattern=r"\d{2}-\d{4}")


for student_id in ["61-38261", "ID61-3826"]:
    print(f"{student_id!r:12} without anchors: {refused(StudentWithoutAnchors, {'id': student_id}) or 'passes':10}"
          f"  with anchors: {refused(Student, {**good_student, 'id': student_id}) or 'passes'}")

'61-38261'   without anchors: passes      with anchors: ['id']
'ID61-3826'  without anchors: passes      with anchors: ['id']


Without anchors both pass: Pydantic's `pattern` looks for a match **anywhere** in the text, like `re.search`, so `61-3826` hidden inside a longer value is enough. `^` (start) and `$` (end) make the whole value match the rule.

#### The booking model

The booking model uses the other three and adds the remaining rules: `event` present, at least one ticket, `payment` and `status` from the brief's lists. The last two rules of the table compare two fields, so they go in one `model_validator`:

- **`total`** equals the sum of `qty × price` over the tickets;
- **`booked_at`** is before the event's date.

Before writing the second one, a decision. `booked_at` is a moment in UTC; the event's `date` is a day with no time. What does "before the event's date" mean for a booking made on that day? We look at how close the bookings in this file come to their event's date, reading both in UTC: `as_utc` converts a time with an offset, and reads a time without one as UTC, as the brief says `booked_at` is.

In [15]:
def as_utc(moment):
    """The same moment in UTC: a time with an offset is converted, a time without one is read as UTC."""
    return moment.replace(tzinfo=dt.timezone.utc) if moment.tzinfo is None else moment.astimezone(dt.timezone.utc)


rows = []
for n, d in docs.items():
    try:
        booked = as_utc(dt.datetime.fromisoformat(at(d, "booked_at")))
        event_day = dt.datetime.combine(dt.date.fromisoformat(at(d, "event", "date")), dt.time(), dt.timezone.utc)
    except (TypeError, ValueError):
        continue  # a value that doesn't parse is the contract's job, not this look's
    rows.append({"line": n, "booking_id": at(d, "booking_id"), "booked_at": at(d, "booked_at"),
                 "event date": at(d, "event", "date"), "hours before the event's date": (event_day - booked) / dt.timedelta(hours=1)})

gaps = pd.DataFrame(rows).sort_values("hours before the event's date", ignore_index=True)
hours = gaps["hours before the event's date"]
print("booked before the event's date:", (hours > 0).sum(),
      "| on the event's date:", ((hours <= 0) & (hours > -24)).sum(),
      "| after the event's date:", (hours <= -24).sum())
gaps.head(6)

booked before the event's date: 278 | on the event's date: 0 | after the event's date: 3


,line,booking_id,booked_at,event date,hours before the event's date
0,224,B0221,2026-11-10T10:00:00Z,2026-11-07,-82.000000
1,117,B0116,2026-11-09T10:00:00Z,2026-11-07,-58.000000
2,170,B0168,2026-10-31T10:00:00Z,2026-10-29,-58.000000
3,90,B0089,2026-10-27T23:48:00Z,2026-10-29,24.200000
4,248,B0245,2026-10-20T23:32:00Z,2026-10-22,24.466667
5,125,B0123,2026-11-03T21:41:00Z,2026-11-05,26.316667


**Decision.** The brief's rule reads *before the event's date*, so a booking must be made on an earlier day than the event, in UTC: `booked_at`'s UTC date `<` `event.date`. A booking made on the event's own date breaks the rule.

The output above shows which bookings fall on or after the event's date and how close the rest come. A gentler reading, *not after the event*, would let bookings on the event's own date through. It can only give different counts if some booking falls on that date, and the first line of the output says whether any does in this file. We keep the brief's words either way, so the second file is judged by the same rule.

**Total.** Money adds up in floats, and `0.1 + 0.2` is not exactly `0.3` in Python. Prices are EGP with at most two decimals, so two amounts less than half a cent apart are the same amount: the rule compares with `math.isclose(..., abs_tol=0.005)` instead of `==`.

**The extra field.** A.0 found `promo_code` in some documents. Pydantic ignores any key a model doesn't declare, without a word, so the model declares it as optional, `str | None = None`: a document without it gets `None` and still passes, because the brief has no rule about it.

Each cross-field rule raises a short message with no numbers in it, so the quarantine can count failures per rule. If both break, the one `ValueError` lists both, separated by `; `.

In [16]:
TOTAL_RULE = "total is not the sum of qty × price"
DATE_RULE = "booked_at is not before the event's date"


class Booking(BaseModel):
    booking_id: str
    booked_at: dt.datetime
    student: Student
    event: Event                                  # required: a missing or null event fails
    tickets: list[Ticket] = Field(min_length=1)  # at least one ticket
    total: float
    payment: Literal["card", "cash", "wallet"]
    status: Literal["confirmed", "cancelled"]
    promo_code: str | None = None                 # only some documents have it (A.0)

    @field_validator("booked_at")
    @classmethod
    def booked_at_in_utc(cls, moment):
        return as_utc(moment)

    @model_validator(mode="after")
    def cross_field_rules(self):
        broken = []
        if not math.isclose(sum(t.qty * t.price for t in self.tickets), self.total, abs_tol=0.005):
            broken.append(TOTAL_RULE)
        if self.booked_at.date() >= self.event.date:
            broken.append(DATE_RULE)
        if broken:
            raise ValueError("; ".join(broken))
        return self

**Probes for the booking model.** The good example is the brief's own document. Each probe breaks one thing, and two probes sit on each side of each cross-field rule: just inside the line, and just outside it.

`problems(data)` lists what Pydantic refuses, as short labels: the path of a field that failed, or the name of each cross-field rule broken.

In [17]:
def problems(data):
    """What Pydantic refuses in a booking: field paths, and the cross-field rules broken. Empty if it passes."""
    try:
        Booking.model_validate(data)
        return []
    except ValidationError as e:
        found = []
        for err in e.errors():
            if err["loc"]:
                found.append(".".join(map(str, err["loc"])))
            else:  # the model validator's ValueError, with every rule it broke
                found += err["msg"].removeprefix("Value error, ").split("; ")
        return sorted(found)


def without(key):
    """The brief's booking with one key removed."""
    return {k: v for k, v in good_booking.items() if k != key}


good_booking ={"booking_id": "B0002", "booked_at": "2026-09-27T12:29:00Z", "student": good_student, "event": good_event,
                "tickets": [good_ticket], "total": 40.0, "payment": "card", "status": "confirmed"}
cents =[{"type": "standard", "qty": 1, "price": 0.1}, {"type": "vip", "qty": 1, "price": 0.2}]

BOOKING_PROBES = [  # (what the probe does, data, the problems Pydantic must find)
    ("the brief's booking", good_booking, []),
    ("with a promo_code", {**good_booking, "promo_code": "TEST10"}, []),
    ("with promo_code null", {**good_booking, "promo_code": None}, []),
    ("without booking_id", without("booking_id"), ["booking_id"]),
    ("without event", without("event"), ["event"]),
    ("event null", {**good_booking, "event": None}, ["event"]),
    ("no tickets", {**good_booking, "tickets": []}, ["tickets"]),
    ("payment 'bank transfer'", {**good_booking, "payment": "bank transfer"}, ["payment"]),
    ("status 'pending'", {**good_booking, "status": "pending"}, ["status"]),
    ("booked_at 'last Tuesday'", {**good_booking, "booked_at": "last Tuesday"}, ["booked_at"]),
    ("inside: total 0.3 for 0.1 + 0.2", {**good_booking, "tickets": cents, "total": 0.3}, []),
    ("outside: total one cent off", {**good_booking, "total": 40.01}, [TOTAL_RULE]),
    ("inside: booked 23:59 UTC the day before", {**good_booking, "booked_at": "2026-10-14T23:59:00Z"}, []),
    ("inside: 01:00 at +03:00 on the day, 22:00 UTC the day before", {**good_booking, "booked_at": "2026-10-15T01:00:00+03:00"}, []),
    ("outside: booked 00:00 UTC on the event's date", {**good_booking, "booked_at": "2026-10-15T00:00:00Z"}, [DATE_RULE]),
    ("outside: booked the day after", {**good_booking, "booked_at": "2026-10-16T10:00:00Z"}, [DATE_RULE]),
    ("both rules broken", {**good_booking, "booked_at": "2026-10-16T10:00:00Z", "total": 50.0}, sorted([TOTAL_RULE, DATE_RULE])),
    ("qty 0 and a wrong total", {**good_booking, "tickets": [{**good_ticket, "qty": 0}], "total": 50.0}, ["tickets.0.qty"]),
]

booking_results = pd.DataFrame([
    {"probe": label, "must find": "; ".join(must) or "(passes)", "Pydantic finds": "; ".join(problems(data)) or "(passes)"}
    for label, data, must in BOOKING_PROBES
])
booking_results["ok"] = booking_results["must find"] == booking_results["Pydantic finds"]
booking_results

,probe,must find,Pydantic finds,ok
0,the brief's booking,(passes),(passes),True
1,with a promo_code,(passes),(passes),True
2,with promo_code null,(passes),(passes),True
3,without booking_id,booking_id,booking_id,True
4,without event,event,event,True
5,event null,event,event,True
6,no tickets,tickets,tickets,True
7,payment 'bank transfer',payment,payment,True
8,status 'pending',status,status,True
9,booked_at 'last Tuesday',booked_at,booked_at,True


In [18]:
# ✓ check
assert booking_results["ok"].all(), booking_results[~booking_results["ok"]]

kept = Booking.model_validate({**good_booking, "promo_code": "TEST10", "seat": "A12"})
assert kept.promo_code == "TEST10" and "seat" not in kept.model_dump()
assert Booking.model_validate(good_booking).promo_code is None
assert kept.booked_at.tzinfo == dt.timezone.utc

print("✓", len(BOOKING_PROBES), "booking probes, and", len(PROBES), "probes for the three small models")
print("  a declared optional field is kept:  promo_code =", repr(kept.promo_code))
print("  an undeclared key is dropped:       'seat' in the model?", "seat" in kept.model_dump())
print("  booked_at is held in UTC:           ", kept.booked_at.isoformat())

✓ 18 booking probes, and 22 probes for the three small models
  a declared optional field is kept:  promo_code = 'TEST10'
  an undeclared key is dropped:       'seat' in the model? False
  booked_at is held in UTC:            2026-09-27T12:29:00+00:00


Three things the probes show:

- **A field rule stops the cross-field rules.** With `qty` 0 *and* a wrong total, Pydantic reports only the `qty`: `mode="after"` runs once every field is valid, so its rules can trust `self.tickets` and `self.event`. A booking is quarantined for the first layer it fails.
- **`==` would be wrong for money.** The booking whose total is `0.3` for `0.1 + 0.2` is correct, and `==` would refuse it. One cent off is refused.
- **The UTC date decides.** `01:00` at `+03:00` on the event's date is `22:00` UTC the day before, so it passes. `00:00` UTC on the event's date fails.

With these four models, every row of the brief's table has a rule and at least one probe.

### A.2 Validate every line

Every line of `DATA` goes through the contract on its own, as in Lab 4's `admit`, so one bad line never stops the others. Each line ends up in exactly one of two places:

- **passed**: its line number and the validated `Booking`;
- **quarantine**: its line number, its `booking_id`, the reasons it failed, and the record as it arrived, so someone can read it, fix the cause and replay it (Lab 4: *rejected is not deleted*).

A line that isn't JSON, or isn't a JSON object, is quarantined too, with `booking_id` `None`.

**Reasons** must read the same every time the same rule breaks, so they can be counted. `reasons_of` keeps the field path without list positions (`tickets.qty`, not `tickets.0.qty`: which ticket broke the rule doesn't change the rule) and Pydantic's message. The model validator's message is split back into one reason per cross-field rule. A line that breaks the same rule twice, with two bad tickets, gets that reason once.

In [19]:
def reasons_of(error):
    """Short reasons from a ValidationError: 'field.path: message', or the cross-field rule broken."""
    reasons = []
    for err in error.errors():
        path = ".".join(str(part) for part in err["loc"] if not isinstance(part, int))
        if path:
            reasons.append(f"{path}: {err['msg']}")
        else:  # the model validator: one message holding every cross-field rule it broke
            reasons += err["msg"].removeprefix("Value error, ").split("; ")
    return list(dict.fromkeys(reasons))  # each reason once, in order


def admit(lines):
    """Validate each line on its own. Returns (passed, quarantine)."""
    passed, quarantine = [], []
    for n, line in enumerate(lines, start=1):
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            quarantine.append({"line": n, "booking_id": None, "reasons": ["line is not valid JSON"], "record": line})
            continue
        if not isinstance(record, dict):
            quarantine.append({"line": n, "booking_id": None, "reasons": ["line is not a JSON object"], "record": record})
            continue
        try:
            passed.append((n, Booking.model_validate(record)))
        except ValidationError as e:
            quarantine.append({"line": n, "booking_id": record.get("booking_id"), "reasons": reasons_of(e), "record": record})
    return passed, quarantine


passed, quarantine = admit(lines)
print("lines read:", len(lines), "| passed:", len(passed), "| quarantined:", len(quarantine))

lines read: 283 | passed: 262 | quarantined: 21


**The quarantine**, one row per line, in file order:

In [20]:
quarantine_table = pd.DataFrame(
    [{"line": q["line"], "booking_id": q["booking_id"], "reasons": "; ".join(q["reasons"])} for q in quarantine],
    columns=["line", "booking_id", "reasons"],
)
quarantine_table

,line,booking_id,reasons
0,11,B0011,"tickets.type: Input should be 'standard', 'vip' or 'student'"
1,16,B0016,tickets.qty: Input should be greater than or equal to 1
2,24,B0024,student.id: String should match pattern '^\d{2}-\d{4}$'
3,50,B0049,event: Input should be a valid dictionary or instance of Event
4,58,B0057,student.id: String should match pattern '^\d{2}-\d{4}$'
5,61,B0060,student.id: String should match pattern '^\d{2}-\d{4}$'
6,91,B0090,"tickets.type: Input should be 'standard', 'vip' or 'student'"
7,102,B0101,total is not the sum of qty × price
8,107,B0106,student.id: String should match pattern '^\d{2}-\d{4}$'
9,117,B0116,booked_at is not before the event's date


**Count per reason.** A line that breaks two rules counts once under each, so the reasons can add up to more than the quarantined lines. The first line of the output says how many lines that is.

In [21]:
per_reason = Counter(reason for q in quarantine for reason in q["reasons"])
print("quarantined lines with more than one reason:", sum(len(q["reasons"]) > 1 for q in quarantine))
pd.Series(per_reason, dtype="int64", name="lines").sort_values(ascending=False).rename_axis("reason").to_frame()

quarantined lines with more than one reason: 0


,lines
reason,
student.id: String should match pattern '^\d{2}-\d{4}$',4
total is not the sum of qty × price,4
"tickets.type: Input should be 'standard', 'vip' or 'student'",3
tickets.qty: Input should be greater than or equal to 1,3
booked_at is not before the event's date,3
event: Input should be a valid dictionary or instance of Event,2
"tickets: List should have at least 1 item after validation, not 0",2


In [22]:
# ✓ check
passed_lines = {n for n, _ in passed}
quarantined_lines = {q["line"] for q in quarantine}
assert len(passed) + len(quarantine) == len(lines), "every line ends up in exactly one place"
assert not passed_lines & quarantined_lines, "no line is both passed and quarantined"
assert all(q["reasons"] for q in quarantine), "every quarantined line has at least one reason"
print("✓", len(lines), "lines =", len(passed), "passed +", len(quarantine), "quarantined, each with its reasons")

✓ 283 lines = 262 passed + 21 quarantined, each with its reasons


**Where did the negative totals go?** A.0 saw a total below zero, and the brief has no rule against that on its own: only that the total equals the sum of `qty × price`. Every line whose total is negative, with what happened to it:

In [23]:
def total_of(record):
    return record.get("total") if isinstance(record, dict) else None


negative = [{"line": q["line"], "booking_id": q["booking_id"], "total": total_of(q["record"]),
             "tickets": q["record"]["tickets"], "reasons": "; ".join(q["reasons"])}
            for q in quarantine
            if isinstance(total_of(q["record"]), (int, float)) and total_of(q["record"]) < 0]
print("passed bookings with a negative total:", sum(b.total < 0 for _, b in passed))
pd.DataFrame(negative, columns=["line", "booking_id", "total", "tickets", "reasons"])

passed bookings with a negative total: 0


,line,booking_id,total,tickets,reasons
0,255,B0252,-10.0,"[{'type': 'student', 'qty': -1, 'price': 10.0}]",tickets.qty: Input should be greater than or equal to 1
1,281,B0278,-30.0,"[{'type': 'standard', 'qty': -1, 'price': 30.0}]",tickets.qty: Input should be greater than or equal to 1


**Reading the quarantine.**

- Every A.0 finding shows up under the rule it breaks: the ticket types outside the list, the `qty` below 1, the ids in other formats, the null events, the empty ticket lists.
- The two cross-field rules appear for the first time. Counting values couldn't show them; the model validator finds the totals that don't add up and the bookings made after the event.
- The negative totals belong to bookings with a negative `qty`, and each one equals its `qty × price`, so the total rule alone would have let them through. The `qty` rule refuses them first, and the model validator never runs on them.
- A null `event` reads *Input should be a valid dictionary or instance of Event*: the key is there, but its value isn't an object. A missing key would read *Field required* (A.1's probes).
- The repeated `booking_id`s from A.0 are not handled here: Pydantic sees one record at a time and can't know another line has the same id. That is A.3.

### A.3 Re-sent bookings

The app sometimes sends the same booking twice. The brief keeps the first copy of each `booking_id` and quarantines the others with the reason `duplicate booking_id`. This step runs on the bookings that passed the contract: a line the contract refused is already in quarantine, for its own reason.

**Look first.** Which `booking_id`s appear more than once among the passed bookings, and are their copies the same booking sent twice? Each copy is shown with its line number, and compared with the first copy as the raw text of its line. The last line of the output says whether any repeated `booking_id` also has a copy in the quarantine.

In [24]:
copies = Counter(b.booking_id for _, b in passed)
repeated = {booking_id for booking_id, n in copies.items() if n > 1}

first_text = {}
rows = []
for n, b in passed:
    if b.booking_id in repeated:
        first_text.setdefault(b.booking_id, lines[n - 1])
        rows.append({"booking_id": b.booking_id, "line": n, "booked_at": b.booked_at, "student": b.student.id,
                     "event": b.event.id, "total": b.total, "status": b.status,
                     "same text as the first copy": lines[n - 1] == first_text[b.booking_id]})

print("passed bookings:", len(passed), "| distinct booking_id:", len(copies), "| booking_ids with copies:", len(repeated))
print("repeated booking_ids that also have a copy in quarantine:",
      sorted(repeated & {q["booking_id"] for q in quarantine}) or "none")
pd.DataFrame(rows, columns=["booking_id", "line", "booked_at", "student", "event", "total", "status",
                            "same text as the first copy"]).sort_values(["booking_id", "line"], ignore_index=True)

passed bookings: 262 | distinct booking_id: 259 | booking_ids with copies: 3
repeated booking_ids that also have a copy in quarantine: none


,booking_id,line,booked_at,student,event,total,status,same text as the first copy
0,B0035,35,2026-11-10 11:33:00+00:00,61-2506,EV-08,135.0,confirmed,True
1,B0035,44,2026-11-10 11:33:00+00:00,61-2506,EV-08,135.0,confirmed,True
2,B0066,67,2026-11-22 06:53:00+00:00,60-6183,EV-08,105.0,confirmed,True
3,B0066,190,2026-11-22 06:53:00+00:00,60-6183,EV-08,105.0,confirmed,True
4,B0107,108,2026-10-27 04:41:00+00:00,61-8559,EV-05,50.0,confirmed,True
5,B0107,121,2026-10-27 04:41:00+00:00,61-8559,EV-05,50.0,confirmed,True


**Keep the first copy.** `keep_first` walks the passed bookings in file order with a set of the ids it has seen. The first copy of each `booking_id` stays; a later copy goes to the quarantine with the reason `duplicate booking_id` and its record as it arrived. `valid` is what the rest of the notebook uses: one validated booking per `booking_id`, with its line number.

In [25]:
def keep_first(passed, lines):
    """Keep the first copy of each booking_id, in file order. Returns (valid, duplicates)."""
    valid, duplicates, seen = [], [], set()
    for n, booking in passed:
        if booking.booking_id in seen:
            duplicates.append({"line": n, "booking_id": booking.booking_id, "reasons": ["duplicate booking_id"],
                               "record": json.loads(lines[n - 1])})
        else:
            seen.add(booking.booking_id)
            valid.append((n, booking))
    return valid, duplicates


valid, duplicates = keep_first(passed, lines)
print("passed:", len(passed), "| kept:", len(valid), "| duplicate copies:", len(duplicates))
pd.DataFrame([{"line": d["line"], "booking_id": d["booking_id"], "reasons": "; ".join(d["reasons"])} for d in duplicates],
             columns=["line", "booking_id", "reasons"])

passed: 262 | kept: 259 | duplicate copies: 3


,line,booking_id,reasons
0,44,B0035,duplicate booking_id
1,121,B0107,duplicate booking_id
2,190,B0066,duplicate booking_id


### Save the quarantine

`quarantine.jsonl` holds every quarantined line, from A.2 and A.3, in file order: one JSON object per line with `line`, `booking_id`, `reasons` and `record`, the line as it arrived. It is written as UTF-8 with `\n` line endings, like `bookings.jsonl`, and opened with `"w"`, so running the notebook again replaces the file instead of adding the same lines twice. Then it is read back and compared with what was meant to be written.

In [26]:
QUARANTINE_FILE = "quarantine.jsonl"

all_quarantined = sorted(quarantine + duplicates, key=lambda q: q["line"])
with open(QUARANTINE_FILE, "w", encoding="utf-8", newline="\n") as f:
    for q in all_quarantined:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

with open(QUARANTINE_FILE, encoding="utf-8") as f:
    read_back = [json.loads(line) for line in f]
assert read_back == all_quarantined, "the file must hold exactly the quarantined lines"
print(QUARANTINE_FILE, "holds", len(read_back), "lines; the first one:")
print(json.dumps(read_back[0], ensure_ascii=False)[:300] if read_back else "(empty)")

quarantine.jsonl holds 24 lines; the first one:
{"line": 11, "booking_id": "B0011", "reasons": ["tickets.type: Input should be 'standard', 'vip' or 'student'"], "record": {"booking_id": "B0011", "booked_at": "2026-10-07T09:43:00Z", "student": {"id": "58-8904", "name": "Karim Shawky", "faculty": "EMS"}, "event": {"id": "EV-01", "title": "Robotics 


### A.4 Report

How many lines were read, how many passed, and how many were quarantined for each reason. A line that breaks two rules counts once under each reason, so the reasons add up to more than the quarantined lines exactly when some line has more than one.

In [27]:
report = pd.Series({
    "lines read": len(lines),
    "refused by the contract (A.2)": len(quarantine),
    "duplicate copies (A.3)": len(duplicates),
    "quarantined lines": len(all_quarantined),
    "passed: one booking per booking_id": len(valid),
}, name="lines")
print(report.to_string(), "\n")

per_reason = Counter(reason for q in all_quarantined for reason in q["reasons"])
pd.Series(per_reason, dtype="int64", name="lines").sort_values(ascending=False).rename_axis("quarantined for").to_frame()

lines read                            283
refused by the contract (A.2)          21
duplicate copies (A.3)                  3
quarantined lines                      24
passed: one booking per booking_id    259 



,lines
quarantined for,
student.id: String should match pattern '^\d{2}-\d{4}$',4
total is not the sum of qty × price,4
tickets.qty: Input should be greater than or equal to 1,3
"tickets.type: Input should be 'standard', 'vip' or 'student'",3
duplicate booking_id,3
booked_at is not before the event's date,3
event: Input should be a valid dictionary or instance of Event,2
"tickets: List should have at least 1 item after validation, not 0",2


In [28]:
# ✓ check
valid_lines = {n for n, _ in valid}
quarantined_lines = {q["line"] for q in all_quarantined}
assert len(lines) == len(valid) + len(all_quarantined), "every line is either passed or quarantined"
assert valid_lines.isdisjoint(quarantined_lines) and valid_lines | quarantined_lines == set(range(1, len(lines) + 1))
assert len({b.booking_id for _, b in valid}) == len(valid), "booking_id is unique among the passed bookings"
assert sum(per_reason.values()) == sum(len(q["reasons"]) for q in all_quarantined)
print("✓", len(lines), "lines read =", len(valid), "passed +", len(all_quarantined), "quarantined;",
      "booking_id unique among the passed")

✓ 283 lines read = 259 passed + 24 quarantined; booking_id unique among the passed


**Reading A.3 and A.4.**

- Every repeated `booking_id` among the passed bookings is the same booking sent twice: each later copy is the first copy's line, character for character. Keeping the first copy loses nothing.
- A copy can arrive far from the first one, as the line numbers in the look show, so the check needs every line seen so far, not just the line before.
- The look also prints whether a repeated `booking_id` had a copy refused by the contract. Whether doing A.3 before A.2 would change any count is tested in Part D's evidence.
- In the report, *passed* means met the contract **and** is the first copy of its `booking_id`. These are the bookings Part B flattens.

## Part B · Flatten

### B.1 Two flat tables at two grains

Each valid booking still nests a student, an event and a list of tickets. Lab 6's rule for flattening: **objects become columns, lists become rows.**

- **`bookings`**: one row per booking. The `student` and `event` objects become columns (`student.id` → `student_id`, `event.club` → `event_club`), and the `tickets` list is dropped from it.
- **`booking_tickets`**: one row per ticket entry, with the `booking_id` it belongs to, `type`, `qty`, `price`, and `amount = qty × price`. The list becomes rows, so the grain changes.

Both start from `valid` (A.3): one validated `Booking` per `booking_id`. `model_dump()` turns each back into a plain dict, with the types the contract gave it (`booked_at` a UTC datetime, `event.date` a date).

In [29]:
dumps = [booking.model_dump() for _, booking in valid]

bookings = pd.json_normalize(dumps, sep="_").drop(columns="tickets")
print(bookings.shape)
bookings.head(3)

(259, 14)

,booking_id,booked_at,total,payment,status,promo_code,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue
0,B0001,2026-09-28 15:46:00+00:00,340.0,card,cancelled,None,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
1,B0002,2026-09-27 12:29:00+00:00,40.0,card,confirmed,None,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
2,B0003,2026-10-13 20:19:00+00:00,60.0,wallet,confirmed,None,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium


In [30]:
booking_tickets = pd.json_normalize(dumps, record_path="tickets", meta=["booking_id"])
booking_tickets = booking_tickets[["booking_id", "type", "qty", "price"]]
booking_tickets["amount"] = booking_tickets["qty"] * booking_tickets["price"]
print(booking_tickets.shape)
booking_tickets.head(3)

(314, 5)


,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0


**Columns, types and grains.** What each table holds, and where each row count comes from: `bookings` against the valid bookings, `booking_tickets` against the number of ticket entries inside them.

In [31]:
print("bookings columns:       ", list(bookings.columns))
print("booking_tickets columns:", list(booking_tickets.columns), "\n")
print(pd.concat([bookings.dtypes, booking_tickets.dtypes], axis=1, keys=["bookings", "booking_tickets"]).fillna("").to_string(), "\n")

ticket_entries = sum(len(booking.tickets) for _, booking in valid)
print("rows of bookings:       ", len(bookings), "| valid bookings:", len(valid))
print("rows of booking_tickets:", len(booking_tickets), "| ticket entries in them:", ticket_entries)
print("ticket entries per booking:", dict(Counter(len(booking.tickets) for _, booking in valid)))

bookings columns:        ['booking_id', 'booked_at', 'total', 'payment', 'status', 'promo_code', 'student_id', 'student_name', 'student_faculty', 'event_id', 'event_title', 'event_club', 'event_date', 'event_venue']
booking_tickets columns: ['booking_id', 'type', 'qty', 'price', 'amount'] 

                            bookings booking_tickets
booking_id                    object          object
booked_at        datetime64[ns, UTC]                
total                        float64                
payment                       object                
status                        object                
promo_code                    object                
student_id                    object                
student_name                  object                
student_faculty               object                
event_id                      object                
event_title                   object                
event_club                    object                
event_date         

**One booking, in both tables.** The first booking with more than one ticket entry: one row in `bookings`, one row per entry in `booking_tickets`. Its `total` lives only on the booking's row.

In [32]:
entries = booking_tickets["booking_id"].value_counts()
several = entries[entries > 1].index
example = bookings.loc[bookings["booking_id"].isin(several), "booking_id"].iloc[0] if len(several) else bookings["booking_id"].iloc[0]

print(bookings.loc[bookings["booking_id"] == example, ["booking_id", "student_id", "event_id", "total", "status"]].to_string(index=False), "\n")
print(booking_tickets[booking_tickets["booking_id"] == example].to_string(index=False))

booking_id student_id event_id  total    status
     B0001    60-3778    EV-01  340.0 cancelled

 

booking_id     type  qty  price  amount
     B0001 standard    1   40.0    40.0
     B0001      vip    3  100.0   300.0


### B.2 Check, with assertions

The brief's two assertions:

1. `booking_id` is unique in `bookings`: one row per booking;
2. for every booking, the `amount`s of its tickets add up to its `total`, compared per booking and to the cent, as in the contract.

The second is checked **per booking**: grand totals that agree could still hide one booking too high and another too low. Each booking's sum is lined up with its total by `booking_id`, so a booking with no ticket rows would show up as a missing sum and fail too.

Two more checks hold the grains to their promise: `bookings` has one row per valid booking, and `booking_tickets` one row per ticket entry. The column lists are as the brief describes them.

In [33]:
# ✓ check
assert bookings["booking_id"].is_unique, "booking_id must be unique in bookings"

ticket_sums = booking_tickets.groupby("booking_id")["amount"].sum()
gap = (ticket_sums.reindex(bookings["booking_id"]).to_numpy() - bookings["total"].to_numpy())
assert pd.notna(gap).all(), "every booking has ticket rows"
assert (abs(gap) < 0.005).all(), "every booking's amounts add up to its total"

assert len(bookings) == len(valid), "bookings: one row per valid booking"
assert len(booking_tickets) == sum(len(booking.tickets) for _, booking in valid), "booking_tickets: one row per ticket entry"
assert "tickets" not in bookings.columns and {"student_id", "event_club"} <= set(bookings.columns)
assert list(booking_tickets.columns) == ["booking_id", "type", "qty", "price", "amount"]

print("✓ booking_id unique in bookings;", len(bookings), "bookings, each with amounts adding up to its total")
print("✓", len(booking_tickets), "ticket rows, one per ticket entry; largest gap between a total and its amounts:", abs(gap).max())

✓ booking_id unique in bookings; 259 bookings, each with amounts adding up to its total
✓ 314 ticket rows, one per ticket entry; largest gap between a total and its amounts: 0.0


**Reading B.1 and B.2.**

- `json_normalize` keeps the plain fields first and adds the flattened `student_*` and `event_*` columns after them. `promo_code` is a column like any other, `None` where a booking has no code (B.3 looks at when the field appears).
- The types the contract gave survive flattening: `booked_at` is a UTC datetime column, `qty` an integer, `price` and `amount` floats. `event_date` holds Python dates, which pandas lists as `object`.
- `booking_tickets` has more rows than `bookings` whenever a booking holds more than one ticket entry. The booking's `total` stays on its one row in `bookings`: copied onto its ticket rows, a two-entry booking would be counted twice in any sum (Lab 6's grain trap; Part D measures it).
- Every booking's amounts add up to its total because the contract refused each booking whose total didn't. B.2 proves the flattening kept that true, booking by booking.

### B.3 Schema drift

The brief warns that some documents have a field its example doesn't. A.0's key sets already showed two shapes of document; now the question is which field, and from when.

**Find it from the data.** `EXAMPLE_KEYS` holds the keys of the brief's example document at each level: the top, `student`, `event`, and each ticket. `extra_fields` returns every key a document has beyond those, as a dotted path, so a new field is found wherever it appears, and a different field in another file is found too.

This runs on the raw parsed documents from A.0, not on `bookings`: once a document is flattened, a missing key and a `null` value look the same. A key counts as present even when its value is `null`.

In [34]:
EXAMPLE_KEYS = {
    "top": {"booking_id", "booked_at", "student", "event", "tickets", "total", "payment", "status"},
    "student": {"id", "name", "faculty"},
    "event": {"id", "title", "club", "date", "venue"},
    "ticket": {"type", "qty", "price"},
}


def extra_fields(doc):
    """The keys a document has that the brief's example doesn't, as dotted paths."""
    found = {key for key in doc if key not in EXAMPLE_KEYS["top"]}
    for part in ["student", "event"]:
        if isinstance(doc.get(part), dict):
            found |= {f"{part}.{key}" for key in doc[part] if key not in EXAMPLE_KEYS[part]}
    for ticket in tickets_of(doc):
        if isinstance(ticket, dict):
            found |= {f"tickets.{key}" for key in ticket if key not in EXAMPLE_KEYS["ticket"]}
    return found


def booked_utc(doc):
    """A document's booked_at as a UTC datetime, or None if it has none that parses."""
    try:
        return as_utc(dt.datetime.fromisoformat(doc["booked_at"]))
    except (KeyError, TypeError, ValueError):
        return None


drift = pd.DataFrame(
    [{"line": n, "booking_id": d.get("booking_id"), "booked_at": booked_utc(d), "extra": extra_fields(d)}
     for n, d in docs.items() if isinstance(d, dict)],
    columns=["line", "booking_id", "booked_at", "extra"],
)
drift["booked_at"] = pd.to_datetime(drift["booked_at"], utc=True)
new_fields = sorted(set().union(*drift["extra"]))

print("fields the brief's example doesn't have:", new_fields or "none")
for field in new_fields:
    print(f"  {field}: in {drift['extra'].apply(lambda extra: field in extra).sum()} of {len(drift)} documents;"
          f" declared in the Booking model? {field in Booking.model_fields}")
print("is the file in booked_at order?", drift["booked_at"].is_monotonic_increasing)

fields the brief's example doesn't have: ['promo_code']
  promo_code: in 193 of 283 documents; declared in the Booking model? True
is the file in booked_at order? False


**From when?** The file is not in time order, so the answer comes from `booked_at`, not from line positions. For each new field: the earliest `booked_at` of a document that has it, and the latest of one that doesn't. If the second comes before the first, the field switched on cleanly at that moment: every document before it lacks the field, every one after has it.

Run twice: on every document that parsed, then on the valid bookings only (A.3's `valid`).

In [35]:
valid_lines = {n for n, _ in valid}
scopes = {"every parsed document": drift, "valid bookings": drift[drift["line"].isin(valid_lines)]}

when = []
for field in new_fields:
    for scope, rows in scopes.items():
        has = rows["extra"].apply(lambda extra: field in extra)
        first_with, last_without = rows.loc[has, "booked_at"].min(), rows.loc[~has, "booked_at"].max()
        when.append({
            "field": field, "documents": scope, "with it": int(has.sum()), "without it": int((~has).sum()),
            "first booking with it": rows.loc[has].sort_values("booked_at")["booking_id"].iloc[0] if has.any() else None,
            "first booked_at with it": first_with, "last booked_at without it": last_without,
            "without it after the first with it": int((~has & (rows["booked_at"] > first_with)).sum()),
            "clean switch": bool(last_without < first_with) if pd.notna(last_without) else True,
        })
pd.DataFrame(when).set_index(["field", "documents"]).T

field                                              promo_code  \
documents                               every parsed document   
with it                                                   193   
without it                                                 90   
first booking with it                                   B0125   
first booked_at with it             2026-10-20 01:05:00+00:00   
last booked_at without it           2026-10-31 10:00:00+00:00   
without it after the first with it                          1   
clean switch                                            False   

field                                                          
documents                                      valid bookings  
with it                                                   173  
without it                                                 86  
first booking with it                                   B0125  
first booked_at with it             2026-10-20 01:05:00+00:00  
last booked_at without it           2026-10-19 18:43:00+00:00  
without it after the first with it                          0  
clean switch                                             True

**The exceptions.** Every document that lacks a new field although it was booked after the field first appeared, with what A.2 and A.3 did with its line:

In [36]:
reasons_by_line = {q["line"]: "; ".join(q["reasons"]) for q in all_quarantined}
exceptions = []
for field in new_fields:
    has = drift["extra"].apply(lambda extra: field in extra)
    late = drift[~has & (drift["booked_at"] > drift.loc[has, "booked_at"].min())]
    for row in late.itertuples():
        exceptions.append({"field": field, "line": row.line, "booking_id": row.booking_id, "booked_at": row.booked_at,
                           "event date": at(docs[row.line], "event", "date"),
                           "quarantined for": reasons_by_line.get(row.line, "(passed)")})
pd.DataFrame(exceptions, columns=["field", "line", "booking_id", "booked_at", "event date", "quarantined for"])

,field,line,booking_id,booked_at,event date,quarantined for
0,promo_code,170,B0168,2026-10-31 10:00:00+00:00,2026-10-29,booked_at is not before the event's date


**What the code does with documents that don't have it.** The `Booking` model declares the field as `promo_code: str | None = None` (A.1). A document without the key passes the contract like any other, because the brief has no rule about the field, and its `promo_code` is `None`. A document that has the key with a `null` value gets `None` as well. In `bookings` the two cases look the same; the raw documents tell them apart:

In [37]:
for field in [f for f in new_fields if f in bookings.columns]:
    had_key = pd.Series(["has the key" if field in docs[n] else "no key" for n, _ in valid], name="raw document")
    in_bookings = bookings[field].isna().map({True: "null", False: "a value"}).rename(f"{field} in bookings")
    print(pd.crosstab(had_key, in_bookings, margins=True, margins_name="all").to_string(), "\n")
    print(f"{field} values in bookings:", dict(Counter(bookings[field].map(repr))))

promo_code in bookings  a value  null  all
raw document                              
has the key                  87    86  173
no key                        0    86   86
all                          87   172  259

 

promo_code values in bookings: {'None': 172, "'CLUB10'": 47, "'EARLY'": 40}


**Reading B.3.**

- **The field** is `promo_code`, the only key beyond the brief's example at any level of the documents. It sits at the top level.
- **From when:** the table gives the first booking that has it and its `booked_at`. Among the valid bookings the switch is clean: every booking before that moment lacks the field, and every one after has it, which is what a new app release adding a field looks like (Lab 6's `app` field).
- **The one exception** among all parsed documents is a line that says it was booked after its own event. A.2 already quarantined it, because its timestamp is impossible, so its place on the wrong side of the switch says nothing about the drift.
- **Documents without the field** pass the contract and stay in every table, with `promo_code` null. A document that sends the key as `null` (no code used) is null too, so in `bookings` a null means either *the app didn't send the field yet* or *no code was used*. The raw documents tell them apart, as the cross-tab shows, and so does `booked_at`: before the switch, the field didn't exist yet. No value is invented, and no booking is dropped for lacking it.